# TP 1 — Partie 4 : prédire les retards de livraison

**Question métier :** au moment où un client passe commande, peut-on prévoir que sa livraison sera en retard ? Si oui, le service client pourrait le prévenir, ou la logistique choisir un autre transporteur.

C'est une **classification binaire** : la cible est `en_retard` (vrai / faux), calculée en Silver.

**Règle d'or :** le modèle ne doit utiliser que des informations **connues au moment de l'achat**. Une variable connue seulement après coup (date de remise au transporteur, délai réel) donnerait un modèle excellent sur le papier mais inutilisable en pratique : c'est une **fuite de données**.

> **⚠️ Un seul notebook actif à la fois.** Chaque notebook ouvert garde sa propre session Spark (jusqu'à 2 Go de mémoire). Avant de commencer, arrêtez le noyau des notebooks précédents : panneau *Running Terminals and Kernels* (barre de gauche) > *Shut Down All*. Les tables restent dans le Lakehouse.

In [ ]:
import pandas as pd
from pyspark.sql import SparkSession, functions as F
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

spark = SparkSession.builder.appName("tp1-ml").getOrCreate()


def a_completer(quoi):
    """Signale clairement une partie du TP qui n'a pas encore été écrite."""
    raise NotImplementedError(f"⚠️ Cellule à compléter : {quoi}")

## Étape 1 — Construire le jeu d'apprentissage depuis le Lakehouse

Les variables viennent de Gold et de Silver. Le calcul est fait par Spark, puis le résultat (environ 90 000 lignes) est converti en DataFrame pandas pour scikit-learn.

In [ ]:
s = lambda t: spark.table(f"lakehouse.silver.{t}")

par_commande = (s("order_items")
    .join(s("products").select("product_id", "product_weight_g", "product_category_en"), "product_id", "left")
    .join(s("sellers").select("seller_id", "seller_state"), "seller_id", "left")
    .groupBy("order_id")
    .agg(F.sum("product_weight_g").alias("poids_total_g"),
         F.first("seller_state").alias("seller_state"),
         F.first("product_category_en").alias("categorie")))

donnees = (spark.table("lakehouse.gold.commandes")
    .filter("en_retard IS NOT NULL")
    .join(par_commande, "order_id", "left")
    .select(
        "order_id", "order_purchase_timestamp",
        "montant_produits", "montant_fret", "nb_articles", "nb_vendeurs", "nb_echeances",
        "poids_total_g", "delai_estime_jours",
        F.month("order_purchase_timestamp").alias("mois_achat"),
        F.dayofweek("order_purchase_timestamp").alias("jour_semaine"),
        "customer_state", "seller_state", "categorie", "mode_paiement",
        (F.col("customer_state") == F.col("seller_state")).cast("int").alias("meme_etat"),
        F.col("en_retard").cast("int").alias("en_retard"))
    .toPandas())

print(donnees.shape)
print("taux de retard :", round(100 * donnees.en_retard.mean(), 1), "%")

## Étape 2 — Découper dans le temps

On apprend sur le passé et on évalue sur le futur, comme en production : apprentissage sur les commandes passées **avant mars 2018**, test sur les suivantes. Un découpage aléatoire mélangerait les périodes et surestimerait la qualité du modèle.

In [ ]:
NUM = ["montant_produits", "montant_fret", "nb_articles", "nb_vendeurs", "nb_echeances",
       "poids_total_g", "delai_estime_jours", "mois_achat", "jour_semaine", "meme_etat"]
CAT = ["customer_state", "seller_state", "categorie", "mode_paiement"]

coupure = pd.Timestamp("2018-03-01")
train = donnees[donnees.order_purchase_timestamp < coupure]
test = donnees[donnees.order_purchase_timestamp >= coupure]
X_train, y_train = train[NUM + CAT], train.en_retard
X_test, y_test = test[NUM + CAT], test.en_retard
print(f"apprentissage : {len(train)} commandes | test : {len(test)} commandes")

## Étape 3 — Un modèle de référence

Exécutez cette cellule, puis répondez à la question.

In [ ]:
naif = (y_test == 0).mean()
print(f"Un modèle qui prédit toujours « à l'heure » a une exactitude de {100 * naif:.1f} %")

> **❓ Question 1.** Ce modèle naïf a une excellente exactitude (*accuracy*) mais il est inutile. Pourquoi ? Quelles métriques faut-il regarder à la place ?

## Étape 4 — Entraîner et évaluer deux modèles

Le prétraitement est fourni : imputation des valeurs manquantes, normalisation des nombres, encodage *one-hot* des catégories.
Complétez la fonction `evaluer` qui entraîne un pipeline et affiche l'AUC ROC et le rapport de classification, puis comparez une régression logistique et une forêt aléatoire (`class_weight="balanced"` pour compenser le déséquilibre).

In [ ]:
pretraitement = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("std", StandardScaler())]), NUM),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("ohe", OneHotEncoder(handle_unknown="ignore", min_frequency=50))]), CAT),
])


def evaluer(nom, modele):
    pipe = Pipeline([("prep", pretraitement), ("modele", modele)])
    # À COMPLÉTER : entraînement, probabilités sur le test, AUC ROC, rapport de classification
    a_completer("le corps de cette fonction")
    return pipe


logreg = evaluer("Régression logistique", a_completer("le modèle à évaluer"))   # À COMPLÉTER
foret = evaluer("Forêt aléatoire", a_completer("le modèle à évaluer"))          # À COMPLÉTER

> **❓ Question 2.** Interprétez les résultats : quel modèle choisiriez-vous ? Le modèle est-il assez bon pour prévenir automatiquement les clients ?

## Étape 5 — Démontrer la fuite de données

Ajoutez la variable `delai_livraison_jours` (le délai **réel**) aux variables, réentraînez la forêt et observez l'AUC.

In [ ]:
# À COMPLÉTER : refaire X_train / X_test avec delai_livraison_jours en plus, réentraîner, afficher l'AUC

> **❓ Question 3.** Pourquoi l'AUC devient-elle presque parfaite ? Pourquoi ce modèle est-il inutilisable ?

## Étape 6 — Ramener les prédictions dans le Lakehouse

Un modèle n'a de valeur que si ses résultats sont exploitables. Écrivez les probabilités de retard de la période de test dans `lakehouse.gold.predictions_retard`, avec les colonnes `order_id`, `customer_state`, `proba_retard`, `en_retard` (valeur réelle). Le tableau de bord les affichera (partie 5).

In [ ]:
# À COMPLÉTER : DataFrame pandas -> spark.createDataFrame(...) -> écriture Iceberg

> **❓ Question 4.** Le modèle a été entraîné une fois, dans un notebook. Que faudrait-il ajouter pour l'utiliser en production ?